# Battery Feature Lab

Read one raw file with BDS, run BFL, inspect the compact result and retrieve core evidence.

In [ ]:
import json
from pathlib import Path

import bds

import bfl

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
input_path = repo_root / "testdataset" / "NCA_k1_0_05C_05degC.xlsx"
if not input_path.is_file():
    raise FileNotFoundError(input_path)
{
    "input_file": str(input_path),
    "bds_source": bds.__file__,
    "bfl_source": bfl.__file__,
}

In [ ]:
bds_frame, bds_report = bds.read_with_report(
    input_path,
    cycler="auto",
    strict=False,
    keep_raw=True,
    current_sign="charge-positive",
    repair_policy="warn",
    time_sampling_policy="warn",
    current_sign_check="none",
)
{
    "rows": bds_frame.height,
    "columns": bds_frame.columns,
    "cycler": bds_report.to_dict()["cycler"],
    "unmapped_columns": bds_report.to_dict()["unmapped_columns"],
    "time_sampling": bds_report.to_dict()["metadata"]["time_sampling"],
    "semantic_sources": bds_report.to_dict()["metadata"]["semantic_sources"],
}

In [ ]:
result = bfl.analyze(
    input_path,
    output_dir=repo_root / "testdataset" / "bfl_outputs" / input_path.stem,
    input_adapter="bds",
    temperature_column="raw:Surface_Temp(degC)",
)
[path.name for path in result.files]

In [ ]:
analysis = json.loads(result.analysis_results_path.read_text(encoding="utf-8"))
metadata = json.loads(result.analysis_metadata_path.read_text(encoding="utf-8"))
validation = json.loads(result.analysis_validation_path.read_text(encoding="utf-8"))
operation_index = analysis["dimensions"]["operation"][0]
{
    "validation_status": validation["status"],
    "output_files": [path.name for path in result.files],
    "output_sizes_bytes": {path.name: path.stat().st_size for path in result.files},
    "dimensions": {
        name: [item["record_type"] for item in items]
        for name, items in analysis["dimensions"].items()
    },
    "operation_sequence": operation_index["attributes"]["operation_sequence"],
    "operation_metrics": operation_index["metrics"],
    "metadata_channels": metadata["channels"],
    "short_window_recomputation": validation["recomputation"]["short_window_recomputation"],
}

In [ ]:
evidence = json.loads(result.analysis_evidence_path.read_text(encoding="utf-8"))
core_types = {
    "response.capacity_aligned_profile",
    "response.current_step_summary",
    "response.relaxation_signature",
}
core_index = {
    item["record_type"]: item
    for item in analysis["dimensions"]["response"]
    if item["record_type"] in core_types
}
core_evidence = {
    record_type: next(
        item
        for item in evidence["records"]
        if item["record_id"] == index["evidence"]["record_id"]
    )
    for record_type, index in core_index.items()
}
{
    record_type: {
        "compact_result": core_index[record_type],
        "source_intervals": record["source_intervals"],
        "method": record["method"],
    }
    for record_type, record in core_evidence.items()
}